# Pipeline Parallelism Tutorial

## Overview

Pipeline parallelism splits model layers across GPUs and uses micro-batching to overlap computation.

### Learning Objectives
- Understand pipeline stages and micro-batches
- Analyze bubble overhead
- Compare GPipe vs PipeDream schedules

### References
- Huang et al., "GPipe: Efficient Training of Giant Neural Networks using Pipeline Parallelism", NeurIPS 2019
- Narayanan et al., "PipeDream: Generalized Pipeline Parallelism for DNN Training", SOSP 2019

## 1. Mathematical Foundation

### 1.1 Pipeline Bubble

With $S$ pipeline stages and $M$ micro-batches:

**Bubble Ratio (GPipe):**
$$\text{Bubble} = \frac{S - 1}{M + S - 1}$$

**Efficiency:**
$$\eta = 1 - \text{Bubble} = \frac{M}{M + S - 1}$$

For high efficiency, need $M >> S$.

### 1.2 Memory Analysis

Each stage stores:
- Model parameters: $\Psi / S$
- Activations: $M \times A$ (for $M$ micro-batches)

Trade-off: More micro-batches → less bubble but more activation memory

## 2. Pipeline Schedules

### GPipe Schedule (Fill-Drain)
```
Time →
Stage 0: [F0][F1][F2][F3]          [B3][B2][B1][B0]
Stage 1:    [F0][F1][F2][F3]       [B3][B2][B1][B0]
Stage 2:       [F0][F1][F2][F3]    [B3][B2][B1][B0]
Stage 3:          [F0][F1][F2][F3] [B3][B2][B1][B0]
                              ↑
                         Bubble time
```

### 1F1B Schedule (Interleaved)
```
Time →
Stage 0: [F0][F1][F2][F3][B0][B1][B2][B3]
Stage 1:    [F0][F1][F2][B0][F3][B1][B2][B3]
Stage 2:       [F0][F1][B0][F2][B1][F3][B2][B3]
Stage 3:          [F0][B0][F1][B1][F2][B2][F3][B3]
```
1F1B reduces peak activation memory from O(M) to O(S)

In [ ]:
import torch
import torch.nn as nn

def calculate_bubble_overhead(num_stages: int, num_microbatches: int):
    """Calculate pipeline bubble overhead."""
    bubble = (num_stages - 1) / (num_microbatches + num_stages - 1)
    efficiency = 1 - bubble
    
    print(f"Stages: {num_stages}, Micro-batches: {num_microbatches}")
    print(f"Bubble ratio: {bubble:.1%}")
    print(f"Efficiency: {efficiency:.1%}")
    return efficiency

# Example: 4 stages, varying micro-batches
for m in [4, 8, 16, 32]:
    calculate_bubble_overhead(4, m)
    print()

## 3. Pipeline Stage Implementation

In [ ]:
class PipelineStage(nn.Module):
    """A single pipeline stage containing multiple layers."""
    
    def __init__(self, layers: nn.ModuleList, stage_id: int):
        super().__init__()
        self.layers = layers
        self.stage_id = stage_id
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x


def partition_model(model: nn.Module, num_stages: int) -> list:
    """Partition model layers into pipeline stages."""
    layers = list(model.children())
    layers_per_stage = len(layers) // num_stages
    
    stages = []
    for i in range(num_stages):
        start = i * layers_per_stage
        end = start + layers_per_stage if i < num_stages - 1 else len(layers)
        stage_layers = nn.ModuleList(layers[start:end])
        stages.append(PipelineStage(stage_layers, i))
    
    return stages

## 4. GPipe Schedule Implementation

In [ ]:
class GPipeScheduler:
    """GPipe-style pipeline scheduler."""
    
    def __init__(self, stages: list, num_microbatches: int):
        self.stages = stages
        self.num_microbatches = num_microbatches
        self.num_stages = len(stages)
    
    def forward(self, inputs: list) -> list:
        """Execute forward passes for all micro-batches."""
        activations = [[None] * self.num_stages for _ in range(self.num_microbatches)]
        
        for mb in range(self.num_microbatches):
            x = inputs[mb]
            for stage_id, stage in enumerate(self.stages):
                x = stage(x)
                activations[mb][stage_id] = x
        
        return activations
    
    def backward(self, activations: list, grad_outputs: list):
        """Execute backward passes in reverse order."""
        for mb in reversed(range(self.num_microbatches)):
            grad = grad_outputs[mb]
            for stage_id in reversed(range(self.num_stages)):
                act = activations[mb][stage_id]
                act.backward(grad)
                grad = act.grad if hasattr(act, 'grad') else None

## 5. Summary

### Key Takeaways

1. **Pipeline Bubble**: Idle time = (S-1)/(M+S-1)
2. **GPipe**: Simple, high activation memory
3. **1F1B**: Lower memory, same bubble
4. **Interleaved**: Further reduces bubble

### Recommendations

| Scenario | Schedule |
|----------|----------|
| Memory constrained | 1F1B |
| Simple implementation | GPipe |
| Minimum bubble | Interleaved 1F1B |